# 04. Pelatihan Model (Model Training)
### Implementasi Aspect-Based Sentiment Analysis (ABSA) Berbasis IndoBERT untuk Pemetaan Keluhan Pada Ulasan Produk Monitor

1. **IndoBERT** (`indobenchmark/indobert-base-p2`) — model utama.
2. **mBERT** (`bert-base-multilingual-cased`) — model pembanding (baseline).

masing-masing untuk dua tugas ABSA (Pontiki et al., 2014):

- **ACD** (*Aspect Category Detection*) — klasifikasi multi-label 9 kategori aspek.
- **ASC** (*Aspect Sentiment Classification*) — klasifikasi sentimen (Positif/Negatif/Netral) per aspek.

> **Catatan penting soal cakupan:** Data uji (`data/splits/test.csv`) **tidak dibuka/dipakai sama sekali** di notebook ini. Sesuai proposal, data tersebut baru digunakan pada tahap *1.1.1.3 Pengujian Model* (notebook terpisah, `05_testing.ipynb`). Notebook ini hanya menggunakan `train.csv` untuk fine-tuning dan `val.csv` untuk pemilihan hyperparameter, checkpoint terbaik, serta kalibrasi threshold ACD — persis seperti yang dijelaskan pada paragraf pembuka Sub-bab 1.1.1.

**Lingkungan yang disarankan:** GPU (mis. Google Colab, T4/A100), karena akan dilakukan *grid search* hyperparameter pada 2 model × 2 tugas.

## 0. Persiapan 

In [1]:
import os
import json
import random
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score, hamming_loss
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

print("Torch version :", torch.__version__)
print("CUDA tersedia :", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device dipakai:", DEVICE)


c:\Users\TECNO\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Torch version : 2.8.0+cpu
CUDA tersedia : False
Device dipakai: cpu


In [2]:
def set_seed(seed: int = 42):
    """Mengunci seed acak agar proses pelatihan dapat direproduksi."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)

In [3]:
BASE_DIR   = Path("..")
DATA_DIR   = BASE_DIR / "data" / "splits"
MODEL_DIR  = BASE_DIR / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
VAL_PATH   = DATA_DIR / "val.csv"

print("Isi model/ saat ini:", os.listdir(MODEL_DIR) if MODEL_DIR.exists() else "belum ada")

Isi model/ saat ini: ['label_maps.json']


## 1. Memuat Data Latih dan Validasi

Data yang dimuat sudah dalam format *long* (satu baris = satu pasangan ulasan-aspek), hasil dari `03_data_preparation.ipynb`. Format ini digunakan langsung untuk tugas ASC, dan akan diagregasi ulang untuk tugas ACD pada bagian berikutnya.

In [4]:
df_train = pd.read_csv(TRAIN_PATH)
df_val   = pd.read_csv(VAL_PATH)

for name, df in [("train", df_train), ("val", df_val)]:
    df["Aspek"] = df["Aspek"].astype(str).str.strip().str.upper()
    df["Sentimen"] = df["Sentimen"].astype(str).str.strip().str.capitalize()

print(f"Baris data latih (long format)    : {len(df_train)}  | ulasan unik: {df_train['Ulasan'].nunique()}")
print(f"Baris data validasi (long format) : {len(df_val)}  | ulasan unik: {df_val['Ulasan'].nunique()}")
df_train.head()

Baris data latih (long format)    : 4794  | ulasan unik: 2659
Baris data validasi (long format) : 603  | ulasan unik: 332


,No,Ulasan,Aspek,Sentimen
0,1,monitornya sangat bagus untuk harga segini bis...,A1,Positif
1,1,monitornya sangat bagus untuk harga segini bis...,C1,Positif
2,1,monitornya sangat bagus untuk harga segini bis...,C2,Positif
3,1,monitornya sangat bagus untuk harga segini bis...,A3,Positif
4,2,cepat akurat direkomendasikan,D1,Positif


### Definisi Ruang Label

Ruang label untuk kedua tugas ditetapkan secara eksplisit dan **disimpan ke `model/label_maps.json`** agar konsisten dipakai kembali pada notebook pengujian (`05_testing.ipynb`) — dengan begitu urutan indeks kelas tidak pernah berubah antar tahap.

In [5]:
ASPECT_LABELS = ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'C1', 'C2', 'D1']  # urutan tetap, dipakai sebagai urutan output Sigmoid (ACD)
SENTIMENT_LABELS = ['Negatif', 'Netral', 'Positif']                     # urutan tetap, dipakai sebagai kelas output ASC

aspect2id    = {a: i for i, a in enumerate(ASPECT_LABELS)}
sentiment2id = {s: i for i, s in enumerate(SENTIMENT_LABELS)}
id2sentiment = {i: s for s, i in sentiment2id.items()}

label_maps = {
    "aspect_labels": ASPECT_LABELS,
    "sentiment_labels": SENTIMENT_LABELS,
    "aspect2id": aspect2id,
    "sentiment2id": sentiment2id,
}
with open(MODEL_DIR / "label_maps.json", "w", encoding="utf-8") as f:
    json.dump(label_maps, f, ensure_ascii=False, indent=2)

print(json.dumps(label_maps, indent=2, ensure_ascii=False))

{
  "aspect_labels": [
    "A1",
    "A2",
    "A3",
    "B1",
    "B2",
    "B3",
    "C1",
    "C2",
    "D1"
  ],
  "sentiment_labels": [
    "Negatif",
    "Netral",
    "Positif"
  ],
  "aspect2id": {
    "A1": 0,
    "A2": 1,
    "A3": 2,
    "B1": 3,
    "B2": 4,
    "B3": 5,
    "C1": 6,
    "C2": 7,
    "D1": 8
  },
  "sentiment2id": {
    "Negatif": 0,
    "Netral": 1,
    "Positif": 2
  }
}


## 2. Penyiapan Data untuk Tugas ACD (Aspect Category Detection)

ACD bersifat multi-label: satu ulasan dapat memiliki lebih dari satu kategori aspek sekaligus. Data long-format karena itu diagregasi kembali per ulasan unik, dan label diubah menjadi vektor multi-hot berdimensi 9 (mengikuti `ASPECT_LABELS`) menggunakan `MultiLabelBinarizer` dengan urutan kelas yang dikunci (`classes=ASPECT_LABELS`).

Vektor multi-hot inilah yang menjadi target bagi lapisan output Sigmoid sebagaimana dijelaskan pada proposal (probabilitas independen 0–1 untuk tiap kategori aspek).

In [6]:
def build_acd_dataframe(df_long: pd.DataFrame) -> pd.DataFrame:
    """Mengagregasi data long-format menjadi satu baris per ulasan unik,
    dengan label berupa himpunan kode aspek yang muncul pada ulasan tsb."""
    grouped = (
        df_long.groupby("Ulasan")["Aspek"]
        .apply(lambda s: sorted(set(s)))
        .reset_index()
        .rename(columns={"Aspek": "aspect_set"})
    )
    return grouped

acd_train_df = build_acd_dataframe(df_train)
acd_val_df   = build_acd_dataframe(df_val)

mlb = MultiLabelBinarizer(classes=ASPECT_LABELS)
mlb.fit([ASPECT_LABELS])  

acd_train_labels = mlb.transform(acd_train_df["aspect_set"]).astype(np.float32)
acd_val_labels   = mlb.transform(acd_val_df["aspect_set"]).astype(np.float32)

print("Jumlah data latih ACD (ulasan unik):", len(acd_train_df))
print("Jumlah data val ACD (ulasan unik)  :", len(acd_val_df))
print("Contoh label multi-hot (baris pertama):", acd_train_labels[0], "->", ASPECT_LABELS)
acd_train_df.head()

Jumlah data latih ACD (ulasan unik): 2659
Jumlah data val ACD (ulasan unik)  : 332
Contoh label multi-hot (baris pertama): [0. 0. 1. 1. 0. 1. 0. 0. 0.] -> ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'C1', 'C2', 'D1']


,Ulasan,aspect_set
0,VESA mount nya kaku dan susah diatur hanya saj...,"[A3, B1, B3]"
1,VESA mount ya biasa saja standar untuk harga s...,"[A3, B2, C2]"
2,VESA mount ya kaku dan susah diatur meski begi...,"[A3, B3]"
3,VESA mount ya terasa murahan tidak sesuai harg...,"[A3, B2, C2]"
4,admin cepat respons banget setiap ditanya soal...,"[B1, C1]"


## 3. Penyiapan Data dan Strategi Input untuk Tugas ASC (*Aspect Sentiment Classification*)

Berbeda dengan tugas ACD yang bersifat *multi-label*, setiap baris data *long-format* pada tahap ini sudah merepresentasikan satu *instance* ASC yang valid, yaitu pasangan tunggal antara ulasan dan aspek dengan satu label sentimen (*single-label*). 

Untuk mengimplementasikan tahap ini, **saya melakukan penyusunan format input data menggunakan skema *sentence-pair classification* standar BERT**, yaitu dengan struktur format sebagai berikut:

$$\text{[CLS]} \ \text{teks\_ulasan} \ \text{[SEP]} \ \text{kode\_aspek} \ \text{[SEP]}$$

Langkah dan implementasi ini diambil berdasarkan poin-poin berikut:

* **Pemenuhan Skenario Pengujian Proposal** <br> Sub-bab 1.1.1.3.1 pada proposal secara eksplisit mensyaratkan dua skenario pengujian ASC. Pertama, menggunakan kode aspek dari ground-truth (Skenario 1). Kedua, menggunakan kode aspek dari hasil prediksi model ACD (Skenario 2). Dengan menginjeksikan kode aspek langsung ke dalam teks input, model dapat mengevaluasi sentimen secara spesifik pada aspek tersebut, terutama ketika satu ulasan yang sama memiliki sentimen berbeda untuk aspek yang berbeda (sesuai poin 2 Identifikasi Masalah).
* **Pemanfaatan Arsitektur Bawaan (Native BERT)** <br> Metode sentence-pair ini memanfaatkan fitur next-sentence input yang sudah tersedia secara bawaan pada arsitektur BERT/IndoBERT (Devlin et al., 2019). Strategi ini saya pilih agar model dapat langsung memahami hubungan semantik antara ulasan dan aspek tanpa perlu menambahkan modul atau parameter baru di luar cakupan model yang telah ditetapkan dalam proposal.
* **Konfigurasi Kepala Klasifikasi (*Classification Head*)** <br>Karena setiap instance pada format ini hanya memiliki satu label sentimen dari total 3 kelas (Positif, Negatif, Netral), saya menggunakan konfigurasi klasifikasi standar dari pustaka `AutoModelForSequenceClassification`. Fungsi aktivasi yang digunakan adalah `softmax` dengan fungsi kerugian `CrossEntropyLoss`. Hal ini membedakannya dengan tahap ACD yang dimodifikasi menggunakan `Sigmoid` (karena bersifat multi-label), sehingga struktur model ASC ini tetap efisien dan native.


In [7]:
def build_asc_dataframe(df_long: pd.DataFrame) -> pd.DataFrame:
    out = df_long[["Ulasan", "Aspek", "Sentimen"]].copy()
    out["label"] = out["Sentimen"].map(sentiment2id)
    assert out["label"].isna().sum() == 0, "Ditemukan label sentimen di luar SENTIMENT_LABELS."
    return out.reset_index(drop=True)

asc_train_df = build_asc_dataframe(df_train)
asc_val_df   = build_asc_dataframe(df_val)

print("Jumlah instance ASC latih:", len(asc_train_df))
print("Jumlah instance ASC val  :", len(asc_val_df))
asc_train_df.head()

Jumlah instance ASC latih: 4794
Jumlah instance ASC val  : 603


,Ulasan,Aspek,Sentimen,label
0,monitornya sangat bagus untuk harga segini bis...,A1,Positif,2
1,monitornya sangat bagus untuk harga segini bis...,C1,Positif,2
2,monitornya sangat bagus untuk harga segini bis...,C2,Positif,2
3,monitornya sangat bagus untuk harga segini bis...,A3,Positif,2
4,cepat akurat direkomendasikan,D1,Positif,2


## 4. Tokenisasi dan Kelas Dataset PyTorch

Tokenizer dipilih sesuai dengan karakteristik model masing-masing (memenuhi Batasan Masalah poin 2):

* **Model IndoBERT:** Menggunakan tokenizer/checkpoint HuggingFace `indobenchmark/indobert-base-p2`
* **Model mBERT:** Menggunakan tokenizer/checkpoint HuggingFace `bert-base-multilingual-cased`

### Justifikasi Penentuan `max_length`
Panjang maksimum token (`max_length`) ditentukan secara **data-driven** menggunakan nilai **persentil ke-95** dari distribusi panjang token pada data latih, serta dibatasi pada nilai batasan maksimum **128**. 

Strategi ini dipilih alih-alih menggunakan nilai yang dipatok kaku (*hardcoded*) demi mencapai efisiensi komputasi. Melalui pendekatan berbasis data ini, ukuran dimensi input model menjadi optimal (mengurangi pemborosan memori akibat *padding* pada teks pendek), namun di sisi lain tetap menjamin bahwa mayoritas ulasan (95% dari total data) dapat dicakup secara utuh tanpa mengalami pemotongan informasi (*truncation*) yang berlebihan.


In [8]:
MODEL_CHECKPOINTS = {
    "indobert": "indobenchmark/indobert-base-p2",
    "mbert": "bert-base-multilingual-cased",
}

def compute_max_length(tokenizer, texts, upper_cap=128, percentile=95):
    lengths = [len(tokenizer.encode(t, add_special_tokens=True)) for t in texts]
    p = int(np.percentile(lengths, percentile))
    return min(max(p, 16), upper_cap)

In [9]:
class ACDDataset(Dataset):
    """Dataset untuk ACD: input = teks ulasan, target = vektor multi-hot (float, utk Sigmoid/BCE)."""
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            list(texts), truncation=True, max_length=max_length, padding=False
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item


class ASCDataset(Dataset):
    """Dataset untuk ASC: input = pasangan (ulasan, kode aspek), target = id sentimen (int, utk CrossEntropy)."""
    def __init__(self, texts_a, texts_b, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            list(texts_a), list(texts_b),
            truncation=True, max_length=max_length, padding=False
        )
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

## 5. Fungsi Metrik untuk Pemantauan Selama Pelatihan

Metrik evaluasi pada tahap ini digunakan secara spesifik untuk **memantau proses pelatihan, memilih *checkpoint* terbaik, serta menentukan kombinasi *hyperparameter* optimal**. Skor yang dihasilkan di sini bukan merupakan nilai akhir laporan penelitian, karena pengujian akhir akan dihitung ulang secara terpisah pada komponen `05_testing.ipynb` menggunakan berkas `test.csv`.

Konsisten dengan ketentuan pada **Proposal Sub-bab 1.1.1.3.2**, metrik utama yang dipantau selama proses ini adalah sebagai berikut:

* **Tugas ACD & ASC:** Memantau skor **Micro-$F_1$** dan **Macro-$F_1$**.
* **Tugas ACD:** Memantau nilai **Hamming Loss** mengingat karakteristik tugas ACD yang bersifat *multi-label classification*.

### Strategi Threshold pada Tugas ACD
Khusus untuk tugas ACD, perhitungan metrik selama proses pelatihan menggunakan nilai ambang batas (*threshold*) sementara sebesar **0.5** pada fungsi aktivasi Sigmoid. Pencarian nilai *threshold* yang benar-benar optimal dilakukan secara terpisah setelah model terbaik berhasil terpilih (dijelaskan pada Bagian 8). Pendekatan ini diambil guna memenuhi pernyataan eksplisit di dalam proposal yang berbunyi: 

> *"Nilai threshold optimal akan ditentukan melalui proses evaluasi pada validation set."*


In [10]:
def compute_metrics_acd(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))       # Sigmoid manual (setara BCEWithLogitsLoss)
    preds = (probs >= 0.5).astype(int)
    labels = labels.astype(int)
    return {
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "hamming_loss": hamming_loss(labels, preds),
    }

def compute_metrics_asc(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
    }

## 6. Fungsi Pembangun Model (*Model Builder Function*)

Fungsi pembangun model ini dirancang untuk menginisialisasi arsitektur kepala klasifikasi (*classification head*) yang sesuai dengan karakteristik dari masing-masing tugas, yaitu ACD dan ASC. 

Pengaturan jenis permasalahan pada pustaka Hugging Face dilakukan secara eksplisit dengan ketentuan sebagai berikut:

* **Tugas ACD (*Aspect Category Detection*):** 
  Konfigurasi disetel menggunakan argumen `problem_type="multi_label_classification"` pada kelas `AutoModelForSequenceClassification`. Pengaturan ini membuat Hugging Face secara otomatis menerapkan fungsi kerugian `BCEWithLogitsLoss`. Fungsi ini merupakan kombinasi yang stabil secara numerik (*numerically stable*) antara fungsi aktivasi Sigmoid dan *Binary Cross-Entropy*. Langkah ini menjadi **implementasi konkret dari instruksi proposal** yang mensyaratkan penggunaan *"fungsi aktivasi Sigmoid pada lapisan output"* untuk tugas *multi-label* ACD.

* **Tugas ASC (*Aspect Sentiment Classification*):** 
  Konfigurasi disetel menggunakan argumen `problem_type="single_label_classification"` (konfigurasi bawaan/*default*). Pengaturan ini menerapkan fungsi kerugian `CrossEntropyLoss` yang setara dengan penggunaan lapisan *softmax* standar pada klasifikasi kelas tunggal (*single-label* per *instance*), sesuai dengan batasan teoritis pada proposal.


In [11]:
def build_model(model_key: str, task: str):
    checkpoint = MODEL_CHECKPOINTS[model_key]
    if task == "acd":
        model = AutoModelForSequenceClassification.from_pretrained(
            checkpoint,
            num_labels=len(ASPECT_LABELS),
            problem_type="multi_label_classification",
        )
    elif task == "asc":
        model = AutoModelForSequenceClassification.from_pretrained(
            checkpoint,
            num_labels=len(SENTIMENT_LABELS),
            problem_type="single_label_classification",
        )
    else:
        raise ValueError(f"Task tidak dikenal: {task}")
    return model.to(DEVICE)

## 7. Pencarian Hyperparameter dan Pelatihan

Sesuai proposal (merujuk rekomendasi Devlin et al., 2019), grid hyperparameter yang diuji:

- **Batch size**: 16, 32
- **Learning rate**: 2e-5, 3e-5, 5e-5
- **Epoch**: 5 (tetap)
- **Optimizer**: AdamW (bawaan `Trainer` HuggingFace)

Total **6 kombinasi** dijalankan untuk **setiap** pasangan (model, tugas) — sehingga 2 model × 2 tugas × 6 kombinasi = **24 proses fine-tuning**. Checkpoint terbaik per kombinasi dipilih berdasarkan `f1_macro` tertinggi pada validation set (`load_best_model_at_end=True`), lalu kombinasi hyperparameter terbaik dipilih dari 6 kandidat tersebut berdasarkan metrik yang sama.

In [12]:
HYPERPARAM_GRID = [
    {"batch_size": bs, "learning_rate": lr, "epochs": 5}
    for bs, lr in itertools.product([16, 32], [2e-5, 3e-5, 5e-5])
]
for i, hp in enumerate(HYPERPARAM_GRID, 1):
    print(f"{i}. {hp}")

1. {'batch_size': 16, 'learning_rate': 2e-05, 'epochs': 5}
2. {'batch_size': 16, 'learning_rate': 3e-05, 'epochs': 5}
3. {'batch_size': 16, 'learning_rate': 5e-05, 'epochs': 5}
4. {'batch_size': 32, 'learning_rate': 2e-05, 'epochs': 5}
5. {'batch_size': 32, 'learning_rate': 3e-05, 'epochs': 5}
6. {'batch_size': 32, 'learning_rate': 5e-05, 'epochs': 5}


In [ ]:
def run_single_training(model_key, task, hp, train_dataset, val_dataset, tokenizer, run_tag):
    """Melatih satu kombinasi (model, task, hyperparameter) dan mengembalikan
    (metrik_val_terbaik, trainer) untuk kombinasi tersebut."""
    set_seed(SEED)
    model = build_model(model_key, task)
    compute_metrics_fn = compute_metrics_acd if task == "acd" else compute_metrics_asc

    tmp_out = MODEL_DIR / "tmp_runs" / run_tag
    args = TrainingArguments(
        output_dir=str(tmp_out),
        per_device_train_batch_size=hp["batch_size"],
        per_device_eval_batch_size=hp["batch_size"],
        learning_rate=hp["learning_rate"],
        num_train_epochs=hp["epochs"],
        optim="adamw_torch",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_strategy="epoch",
        report_to="none",
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics_fn,
    )

    trainer.train()
    val_metrics = trainer.evaluate()
    return val_metrics, trainer

### 7.1 Menjalankan Grid Search per (Model, Tugas)

Loop berikut mengiterasi seluruh kombinasi (model × tugas × hyperparameter), mencatat metrik validasi tiap kombinasi ke `hyperparam_search_results.csv`, lalu menyimpan **hanya** checkpoint dengan `f1_macro` tertinggi ke `model/{model}_{task}/best_model/`.

In [ ]:
all_results = []  # log seluruh kombinasi, utk dokumentasi & lampiran 
best_trainers = {} # menyimpan trainer terbaik per (model_key, task) sebelum disimpan ke disk

TASK_DATASETS = {} # cache dataset per (model_key, task) karena tokenizer berbeda per model

for model_key, checkpoint in MODEL_CHECKPOINTS.items():
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)

    max_len_acd = compute_max_length(tokenizer, acd_train_df["Ulasan"].tolist())
    acd_train_ds = ACDDataset(acd_train_df["Ulasan"], acd_train_labels, tokenizer, max_len_acd)
    acd_val_ds   = ACDDataset(acd_val_df["Ulasan"],   acd_val_labels,   tokenizer, max_len_acd)

    max_len_asc = compute_max_length(tokenizer, asc_train_df["Ulasan"].tolist())
    asc_train_ds = ASCDataset(asc_train_df["Ulasan"], asc_train_df["Aspek"], asc_train_df["label"], tokenizer, max_len_asc)
    asc_val_ds   = ASCDataset(asc_val_df["Ulasan"],   asc_val_df["Aspek"],   asc_val_df["label"],   tokenizer, max_len_asc)

    TASK_DATASETS[model_key] = {
        "acd": (acd_train_ds, acd_val_ds, max_len_acd),
        "asc": (asc_train_ds, asc_val_ds, max_len_asc),
    }

    for task in ["acd", "asc"]:
        train_ds, val_ds, max_len = TASK_DATASETS[model_key][task]
        best_f1_macro = -1
        best_hp = None
        best_trainer_obj = None

        for hp in HYPERPARAM_GRID:
            run_tag = f"{model_key}_{task}_bs{hp['batch_size']}_lr{hp['learning_rate']}"
            print(f"\n=== Training: {run_tag} ===")

            val_metrics, trainer = run_single_training(
                model_key, task, hp, train_ds, val_ds, tokenizer, run_tag
            )

            row = {"model": model_key, "task": task, **hp, **val_metrics}
            all_results.append(row)

            f1m = val_metrics.get("eval_f1_macro", -1)
            if f1m > best_f1_macro:
                best_f1_macro = f1m
                best_hp = hp
                best_trainer_obj = trainer

        best_trainers[(model_key, task)] = {
            "trainer": best_trainer_obj,
            "hp": best_hp,
            "f1_macro": best_f1_macro,
            "tokenizer": tokenizer,
            "max_length": max_len,
        }
        print(f">>> Terbaik utk ({model_key}, {task}): {best_hp} | f1_macro(val) = {best_f1_macro:.4f}")

results_df = pd.DataFrame(all_results)
results_df.to_csv(MODEL_DIR / "hyperparam_search_results.csv", index=False)
results_df

c:\Users\TECNO\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\TECNO\.cache\huggingface\hub\models--indobenchmark--indobert-base-p2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



=== Training: indobert_acd_bs16_lr2e-05 ===


### 7.2 Visualisasi Kurva Validasi

Proposal mencatat observasi awal bahwa "nilai hyperparameter yang terlalu besar cenderung menurunkan kurva validasi, mengindikasikan overfitting." Grafik berikut membandingkan `f1_macro` validasi akhir antar kombinasi hyperparameter untuk memverifikasi pola tersebut secara visual pada dataset ini.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, task in zip(axes, ["acd", "asc"]):
    sub = results_df[results_df["task"] == task].copy()
    sub["config"] = sub.apply(lambda r: f"bs{int(r['batch_size'])}_lr{r['learning_rate']}", axis=1)
    for model_key in MODEL_CHECKPOINTS:
        s = sub[sub["model"] == model_key].sort_values("config")
        ax.plot(s["config"], s["eval_f1_macro"], marker="o", label=model_key)
    ax.set_title(f"Macro-F1 Validasi — Tugas {task.upper()}")
    ax.set_xlabel("Konfigurasi (batch size, learning rate)")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
axes[0].set_ylabel("Macro-F1 (validasi)")
plt.tight_layout()
plt.show()

## 8. Kalibrasi Threshold Optimal (Khusus ACD)

Sesuai penjelasan pembuka Sub-bab 1.1.1, threshold konversi probabilitas Sigmoid → label biner ditentukan lewat pencarian pada validation set yang memaksimalkan F1-Score. Pencarian dilakukan **per model** (IndoBERT dan mBERT dapat memiliki threshold optimal berbeda) dengan menyisir kandidat 0.05–0.95.

In [ ]:
def search_best_threshold(trainer, val_dataset, labels_true, thresholds=np.arange(0.05, 0.96, 0.05)):
    pred_output = trainer.predict(val_dataset)
    probs = 1 / (1 + np.exp(-pred_output.predictions))

    best_t, best_f1 = 0.5, -1
    scores = []
    for t in thresholds:
        preds = (probs >= t).astype(int)
        f1m = f1_score(labels_true, preds, average="macro", zero_division=0)
        scores.append((t, f1m))
        if f1m > best_f1:
            best_f1, best_t = f1m, t
    return best_t, best_f1, scores

optimal_thresholds = {}
for model_key in MODEL_CHECKPOINTS:
    entry = best_trainers[(model_key, "acd")]
    _, acd_val_ds_m, _ = TASK_DATASETS[model_key]["acd"]
    best_t, best_f1, scores = search_best_threshold(entry["trainer"], acd_val_ds_m, acd_val_labels)
    optimal_thresholds[model_key] = {"threshold": float(best_t), "f1_macro_at_threshold": float(best_f1)}
    print(f"{model_key}: threshold optimal = {best_t:.2f} (Macro-F1 val = {best_f1:.4f})")

with open(MODEL_DIR / "acd_optimal_thresholds.json", "w", encoding="utf-8") as f:
    json.dump(optimal_thresholds, f, ensure_ascii=False, indent=2)

## 9. Menyimpan Model dan Artefak Terbaik

Checkpoint terbaik per (model, tugas) — beserta tokenizer-nya — disimpan permanen ke `model/{model}_{task}/best_model/`. Ringkasan hyperparameter terpilih dan skor validasi disimpan sebagai `model/training_summary.json`, agar dapat dirujuk langsung pada narasi bab hasil penelitian maupun pada notebook pengujian berikutnya.

In [ ]:
training_summary = {}

for (model_key, task), entry in best_trainers.items():
    save_dir = MODEL_DIR / f"{model_key}_{task}" / "best_model"
    save_dir.mkdir(parents=True, exist_ok=True)

    entry["trainer"].save_model(str(save_dir))
    entry["tokenizer"].save_pretrained(str(save_dir))

    training_summary[f"{model_key}_{task}"] = {
        "checkpoint_source": MODEL_CHECKPOINTS[model_key],
        "best_hyperparameters": entry["hp"],
        "f1_macro_val": entry["f1_macro"],
        "max_length": entry["max_length"],
        "saved_path": str(save_dir),
    }
    print(f"Tersimpan: {save_dir}")

with open(MODEL_DIR / "training_summary.json", "w", encoding="utf-8") as f:
    json.dump(training_summary, f, ensure_ascii=False, indent=2)

print("\nRingkasan pelatihan:")
print(json.dumps(training_summary, indent=2, ensure_ascii=False))

In [ ]:
# Membersihkan direktori checkpoint sementara agar tidak memenuhi penyimpanan
import shutil
tmp_dir = MODEL_DIR / "tmp_runs"
if tmp_dir.exists():
    shutil.rmtree(tmp_dir)
    print("Checkpoint sementara (tmp_runs/) dibersihkan.")

## 10. Ringkasan Tahap Pelatihan

Pada tahap ini telah dihasilkan:

1. `model/label_maps.json` — definisi ruang label ACD & ASC (dipakai ulang di tahap pengujian).
2. `model/hyperparam_search_results.csv` — log lengkap 24 kombinasi (2 model × 2 tugas × 6 hyperparameter) untuk lampiran skripsi.
3. `model/acd_optimal_thresholds.json` — threshold Sigmoid optimal per model untuk ACD.
4. `model/{indobert,mbert}_{acd,asc}/best_model/` — 4 checkpoint model hasil fine-tuning terbaik.
5. `model/training_summary.json` — ringkasan hyperparameter & skor validasi terbaik per model/tugas.